# Notebook 03: Post-Training Analysis

**Memory x RL Interaction Experiment**

This notebook:
1. Loads all results from Modal volume
2. Runs statistical tests (McNemar's, bootstrap CI, Cohen's g)
3. Generates all 5 figures + tactic heatmap
4. Determines which what-if outcome matches results

In [ ]:
import json
import os
import sys
sys.path.insert(0, '..')

import numpy as np

from lib.config import ExperimentConfig
from lib.analysis import (
    bootstrap_ci, mcnemar_test, holm_bonferroni, cohens_g,
    run_all_comparisons, generate_summary_report,
    plot_training_curves, plot_eval_accuracy, plot_absorption_delta,
    plot_playbook_evolution, plot_answer_entropy, plot_tactic_heatmap,
    CONDITION_LABELS,
)

CFG = ExperimentConfig()

# Results directory (download from Modal first)
RESULTS_DIR = "../../../results"  # Adjust path as needed
FIGURES_DIR = "./figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

## 1. Load Results

In [ ]:
# Load per-condition evaluation results
conditions = ["B", "C", "C-abs", "D", "D-abs", "E"]
seeds = CFG.SEEDS

eval_results = {}  # {condition: {seed: {dataset: [results]}}}
training_results = {}  # {condition: {seed: {epoch_metrics: [...]}}}

for cond in conditions:
    eval_results[cond] = {}
    for seed in seeds:
        result_dir = os.path.join(RESULTS_DIR, "eval", cond, f"seed{seed}")
        summary_path = os.path.join(result_dir, "summary.json")
        if os.path.exists(summary_path):
            with open(summary_path) as f:
                eval_results[cond][seed] = json.load(f)
            print(f"  Loaded {cond} seed={seed}: {eval_results[cond][seed].get('overall_accuracy', 'N/A')}")
        else:
            print(f"  Missing: {cond} seed={seed}")

# Load training metrics
for cond in ["B", "C", "D", "E"]:
    training_results[cond] = {}
    for seed in seeds:
        done_path = os.path.join(RESULTS_DIR, cond, f"seed{seed}", "training_done.json")
        if os.path.exists(done_path):
            with open(done_path) as f:
                training_results[cond][seed] = json.load(f)

print(f"\nLoaded results for {len(eval_results)} conditions")

## 2. Aggregate Binary Outcomes (Across Seeds)

In [ ]:
# Aggregate per-problem binary outcomes across seeds via majority vote
# For each (condition, problem), count correct across seeds

binary_outcomes = {}  # {condition: [True/False per problem]}

for cond in conditions:
    # Collect per-dataset results across all seeds
    all_ds_results = {}  # {dataset: {problem_id: [correct_per_seed]}}
    
    for seed in seeds:
        if seed not in eval_results.get(cond, {}):
            continue
        
        seed_dir = os.path.join(RESULTS_DIR, "eval", cond, f"seed{seed}")
        for ds_name in ["aime_2025", "olympmath_easy", "amc12"]:
            ds_path = os.path.join(seed_dir, f"{ds_name}.json")
            if not os.path.exists(ds_path):
                continue
            with open(ds_path) as f:
                ds_results = json.load(f)
            
            if ds_name not in all_ds_results:
                all_ds_results[ds_name] = {}
            
            for r in ds_results:
                pid = r["id"]
                if pid not in all_ds_results[ds_name]:
                    all_ds_results[ds_name][pid] = []
                all_ds_results[ds_name][pid].append(r["correct"])
    
    # Aggregate: majority vote across seeds
    outcomes = []
    for ds_name in sorted(all_ds_results.keys()):
        for pid in sorted(all_ds_results[ds_name].keys()):
            seed_results = all_ds_results[ds_name][pid]
            # Majority vote: correct if >50% of seeds got it right
            outcomes.append(sum(seed_results) > len(seed_results) / 2)
    
    binary_outcomes[cond] = outcomes
    if outcomes:
        print(f"{cond}: {sum(outcomes)}/{len(outcomes)} = {sum(outcomes)/len(outcomes):.1%}")
    else:
        print(f"{cond}: no results")

## 3. Statistical Tests

In [ ]:
# Define comparison pairs
comparison_pairs = [
    ("D", "B", "D (Active) vs B (GRPO-only)"),
    ("D", "C", "D (Active) vs C (Static)"),
    ("C", "B", "C (Static) vs B (GRPO-only)"),
    ("E", "B", "E (Novelty) vs B (GRPO-only)"),
    ("D", "D-abs", "D vs D-abs (absorption)"),
    ("C", "C-abs", "C vs C-abs (absorption)"),
    ("D", "E", "D (Active) vs E (Novelty)"),
]

# Filter to conditions with data
available = [c for c in conditions if binary_outcomes.get(c)]
comparison_pairs = [
    (a, b, label) for a, b, label in comparison_pairs
    if a in available and b in available
]

if comparison_pairs:
    analysis = run_all_comparisons(binary_outcomes, comparison_pairs)
    report = generate_summary_report({}, binary_outcomes, comparison_pairs)
    print(report)
else:
    print("No comparison pairs available (need eval results first)")

## 4. Table 1: Accuracy

In [ ]:
print("Table 1: Evaluation Accuracy (Majority Vote Across Seeds)")
print("=" * 60)
print(f"{'Condition':20s} {'Accuracy':>10s} {'95% CI':>20s} {'N':>6s}")
print("-" * 60)

for cond in conditions:
    outcomes = binary_outcomes.get(cond, [])
    if outcomes:
        mean, lo, hi = bootstrap_ci(outcomes)
        label = CONDITION_LABELS.get(cond, cond)
        print(f"{label:20s} {mean:10.1%} [{lo:.1%}, {hi:.1%}] {len(outcomes):6d}")
    else:
        print(f"{CONDITION_LABELS.get(cond, cond):20s} {'N/A':>10s}")

## 5. Table 2: Pairwise Comparisons

In [ ]:
if comparison_pairs:
    print("Table 2: Pairwise Comparisons (McNemar's, Holm-Bonferroni)")
    print("=" * 80)
    print(f"{'Comparison':35s} {'p-raw':>8s} {'p-adj':>8s} {'g':>6s} {'n01':>4s} {'n10':>4s} {'Sig':>5s}")
    print("-" * 80)
    
    for r in analysis["comparisons"]:
        sig = "***" if r["p_adjusted"] < 0.001 else "**" if r["p_adjusted"] < 0.01 else "*" if r["p_adjusted"] < 0.05 else "ns"
        print(
            f"{r['label']:35s} "
            f"{r['p_value']:8.4f} "
            f"{r['p_adjusted']:8.4f} "
            f"{r['cohens_g']:6.3f} "
            f"{r['n01']:4d} "
            f"{r['n10']:4d} "
            f"{sig:>5s}"
        )
else:
    print("No comparisons available")

## 6. Figure 1: Training Curves

In [ ]:
# Aggregate training metrics across seeds (mean)
agg_training = {}
for cond in ["B", "C", "D", "E"]:
    all_metrics = []
    for seed in seeds:
        if seed in training_results.get(cond, {}):
            metrics = training_results[cond][seed].get("epoch_metrics", [])
            if metrics:
                all_metrics.append(metrics)
    
    if all_metrics:
        # Average across seeds
        n_epochs = min(len(m) for m in all_metrics)
        avg_metrics = []
        for e in range(n_epochs):
            avg = {"epoch": e}
            for key in ["answer_entropy", "pb_size", "pb_entropy"]:
                vals = [m[e].get(key, 0) for m in all_metrics]
                avg[key] = np.mean(vals)
            avg_metrics.append(avg)
        agg_training[cond] = {"epoch_metrics": avg_metrics}

if agg_training:
    plot_training_curves(agg_training, os.path.join(FIGURES_DIR, "fig1_training_curves.png"))
else:
    print("No training metrics available for Figure 1")

## 7. Figure 2: Eval Accuracy Bar Chart

In [ ]:
if any(binary_outcomes.get(c) for c in conditions):
    accuracy_table = {}
    for cond in conditions:
        outcomes = binary_outcomes.get(cond, [])
        if outcomes:
            mean, lo, hi = bootstrap_ci(outcomes)
            accuracy_table[cond] = {"mean": mean, "ci_lower": lo, "ci_upper": hi}
    
    plot_eval_accuracy(accuracy_table, os.path.join(FIGURES_DIR, "fig2_eval_accuracy.png"))
else:
    print("No eval results available for Figure 2")

## 8. Figure 3: Absorption Delta

In [ ]:
if all(c in accuracy_table for c in ["C", "C-abs", "D", "D-abs"]):
    plot_absorption_delta(accuracy_table, os.path.join(FIGURES_DIR, "fig3_absorption.png"))
else:
    print("Need C, C-abs, D, D-abs results for absorption plot")

## 9. Figure 4: Playbook Evolution (Condition D)

In [ ]:
if "D" in agg_training:
    plot_playbook_evolution(agg_training, os.path.join(FIGURES_DIR, "fig4_playbook_evolution.png"))
else:
    print("No Condition D training metrics for playbook evolution")

## 10. Figure 5: Answer Entropy (Collapse Detection)

In [ ]:
if agg_training:
    plot_answer_entropy(agg_training, os.path.join(FIGURES_DIR, "fig5_answer_entropy.png"))
else:
    print("No training metrics for answer entropy plot")

## 11. Tactic Diversity Analysis (Optional - requires Kimi API)

In [ ]:
# This cell requires Kimi API access and eval solution texts
# Uncomment and modify paths as needed

# from lib.analysis import tactic_distribution, shannon_entropy, plot_tactic_heatmap
# 
# tactic_data = {}
# for cond in ["B", "C", "D", "E"]:
#     # Load solutions from eval results
#     solutions = []  # Collect solution texts
#     for seed in seeds:
#         ds_path = os.path.join(RESULTS_DIR, "eval", cond, f"seed{seed}", "aime_2025.json")
#         if os.path.exists(ds_path):
#             with open(ds_path) as f:
#                 for r in json.load(f):
#                     if r.get("correct"):
#                         solutions.append(r.get("raw", r.get("predicted", "")))
#     
#     if solutions:
#         tactic_data[cond] = tactic_distribution(solutions[:50])  # Limit API calls
#         entropy = shannon_entropy(tactic_data[cond])
#         print(f"{cond}: entropy={entropy:.2f}, distribution={tactic_data[cond]}")
# 
# if tactic_data:
#     plot_tactic_heatmap(tactic_data, os.path.join(FIGURES_DIR, "fig6_tactic_heatmap.png"))

## 12. Verdict: What-If Outcome

In [ ]:
if binary_outcomes.get("D") and binary_outcomes.get("B"):
    d_acc = sum(binary_outcomes["D"]) / len(binary_outcomes["D"])
    b_acc = sum(binary_outcomes["B"]) / len(binary_outcomes["B"])
    gap = d_acc - b_acc
    
    print("=" * 60)
    print("VERDICT")
    print("=" * 60)
    print(f"  B (GRPO-only):     {b_acc:.1%}")
    print(f"  D (Active PB+GRPO): {d_acc:.1%}")
    print(f"  Gap (D - B):        {gap:+.1%}")
    print()
    
    if gap > 0.10:
        verdict = "CO-EVOLUTION WINS"
        next_step = "Scale up: more epochs, larger model, add strong-model verifier."
    elif gap > 0.05:
        verdict = "MARGINAL SYNERGY"
        next_step = "Try: more epochs, verifier-filtered reward, archetype-anchored curriculum."
    elif gap > -0.05:
        verdict = "GRPO SUBSUMES PLAYBOOK"
        next_step = "Weight updates capture what playbook provides. Focus on TTRL improvements."
    else:
        verdict = "PLAYBOOK INTERFERES WITH RL"
        next_step = "Investigate reward noise from reflect/curate, or playbook poisoning under RL."
    
    print(f"  VERDICT: {verdict}")
    print(f"  NEXT:    {next_step}")
    
    # Check absorption
    if binary_outcomes.get("D-abs"):
        dabs_acc = sum(binary_outcomes["D-abs"]) / len(binary_outcomes["D-abs"])
        absorption = d_acc - dabs_acc
        print(f"\n  Absorption (D - D-abs): {absorption:+.1%}")
        if absorption > 0.05:
            print("  -> Playbook provides value beyond what weights absorbed")
        elif absorption > -0.02:
            print("  -> Weights fully absorbed playbook knowledge")
        else:
            print("  -> Playbook may be interfering at eval time")
else:
    print("Need D and B results to determine verdict")

## 13. Save Analysis

In [ ]:
analysis_output = {
    "accuracy": {
        cond: {
            "mean": sum(o) / len(o) if o else 0,
            "n": len(o),
        }
        for cond, o in binary_outcomes.items()
        if o
    },
}

if comparison_pairs:
    analysis_output["comparisons"] = analysis["comparisons"]

with open(os.path.join(FIGURES_DIR, "analysis_results.json"), "w") as f:
    json.dump(analysis_output, f, indent=2, default=str)

print("Analysis saved to figures/analysis_results.json")